# Phase 4 — Unified Classification

Phase 2 found candidates in the main Chordopoxvirinae search pool; Phase 3
found hits in the orphan-taxa pool. This phase pools **every** candidate,
regardless of which search found it, into one set and passes them all through
the same classification — closing the inconsistency where orphan hits would
otherwise skip the paralog/fragment screening applied to the main candidates.

Classification is driven entirely by curated Pfam families and one length rule,
so every decision has a citeable basis. Filters are applied in order:

1. **PF06086 present** → A26 paralog. PF06086 is the *Orthopoxvirus A26L/A30L
   protein* family; A26 carries its own distinct domain, so a match is decisive.
2. **PF04508 present** → A-type inclusion (ATI) protein. PF04508 is the
   *Pox_A_type_inc* (A-type inclusion protein repeat) family — a curated HMM
   that replaces the earlier hand-built ATI regex.
3. **Too long** (> `length_threshold`) → A26 paralog. A29L is a short protein
   (~110 aa); a much longer sequence is more likely A26 or a fusion. VACV A26
   (~58 kDa) versus A27/A29 (~12 kDa) motivates this cutoff.

Anything passing all three and carrying the **PF02346** domain (*Pox_A27*, the
A27/A29 family) is a true **A29L ortholog**.

The A29L C-terminal motif is retained only as a per-sequence **annotation**
(logged in the output, never used to include or exclude a sequence), since it
is an in-house pattern derived from the query alignment rather than a curated,
citeable signature. Keeping it visible preserves the biological signal without
resting any classification decision on an unpublished pattern.

**Steps**

0. Setup & configuration
1. Prerequisites — required files
2. Pool Phase 2 candidates + Phase 3 orphan hits
3. Domain scan the pooled set (PF02346, PF06086, PF04508)
4. Classify & summarize
5. Write true ortholog sequences

## Step 0 — Setup & configuration

Three HMM profiles are downloaded manually from InterPro/Pfam and placed beside
this notebook: `PF02346.hmm` (Pox_A27, the A27/A29 family), `PF06086.hmm`
(Orthopoxvirus A26L/A30L), and `PF04508.hmm` (Pox_A_type_inc, the ATI repeat).
Record the InterPro/Pfam release version used, since family names can change
between releases even though accessions are stable.

The A29L C-terminal `MOTIF` is compiled here but used only as a logged
annotation, not a filter. External tools run through the shared `run_command`
helper, so every invocation is logged and a non-zero exit raises a clear error;
FASTA and domain-table parsing use the shared `seqio_utils` module.

In [1]:
from __future__ import annotations

import json
import logging
import re
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

from seqio_utils import parse_fasta_by_accession, parse_domtblout

In [2]:
@dataclass(frozen=True)
class Config:
    pf02346_hmm: Path = Path("PF02346.hmm")
    pf06086_hmm: Path = Path("PF06086.hmm")
    pf04508_hmm: Path = Path("PF04508.hmm")

    phase2_candidates_fasta: Path = Path("output_phase2/homology_candidates.fasta")
    phase3_orphan_hits_fasta: Path = Path("output_phase3/orphan_hits.fasta")

    wsl_prefix: list[str] = field(default_factory=lambda: ["wsl"])

    length_threshold: int = 250
    motif_pattern: str = r"[KR]?KID[VM]QTG"

    out_dir: Path = field(default_factory=lambda: Path("output_phase4"))

    @property
    def combined_candidates_fasta(self) -> Path:
        return self.out_dir / "combined_candidates.fasta"

    @property
    def domtbl_pf02346(self) -> Path:
        return self.out_dir / "cand_PF02346.domtbl"

    @property
    def domtbl_pf06086(self) -> Path:
        return self.out_dir / "cand_PF06086.domtbl"

    @property
    def domtbl_pf04508(self) -> Path:
        return self.out_dir / "cand_PF04508.domtbl"

    @property
    def classified_json(self) -> Path:
        return self.out_dir / "candidates_classified.json"

    @property
    def true_orthologs_fasta(self) -> Path:
        return self.out_dir / "true_orthologs.fasta"


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "phase4.log"),
    ],
)
log = logging.getLogger("phase4")

MOTIF = re.compile(CFG.motif_pattern)

In [3]:
class CommandError(RuntimeError):
    """Raised when an external tool exits with a non-zero status."""


def run_command(cmd: list[str], log_file: Path) -> None:
    log.info("Running: %s", " ".join(cmd))
    with open(log_file, "w", encoding="utf-8") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise CommandError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}. "
            f"See {log_file} for details."
        )

## Step 1 — Prerequisites: required files

Verifies every dependency exists before any tool launches: the three `.hmm`
profiles (downloaded manually from InterPro/Pfam), Phase 2's
`homology_candidates.fasta`, and Phase 3's `orphan_hits.fasta`. Failing fast
with a clear message is far easier to diagnose than a cryptic tool error later.

In [4]:
required = [
    CFG.pf02346_hmm, CFG.pf06086_hmm, CFG.pf04508_hmm,
    CFG.phase2_candidates_fasta, CFG.phase3_orphan_hits_fasta,
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required input file(s): {[str(p) for p in missing]}. "
        "PF02346.hmm, PF06086.hmm and PF04508.hmm must be downloaded manually "
        "from InterPro/Pfam."
    )
log.info("All prerequisite files found.")

2026-07-22 11:28:36,818 [INFO] All prerequisite files found.


## Step 2 — Pool Phase 2 candidates + Phase 3 orphan hits

Merges the two candidate FASTAs into one pooled set. If the same accession
appears in both — which should not happen, since the two source pools are
taxonomically disjoint — it is logged as a warning and kept once, taking the
Phase 2 sequence. Checkpointed on `combined_candidates.fasta`.

In [5]:
if CFG.combined_candidates_fasta.exists():
    log.info("%s exists — loading pooled candidates.", CFG.combined_candidates_fasta)
    pooled = parse_fasta_by_accession(CFG.combined_candidates_fasta)
else:
    log.info("Pooling Phase 2 candidates and Phase 3 orphan hits ...")
    phase2_seqs = parse_fasta_by_accession(CFG.phase2_candidates_fasta)
    phase3_seqs = parse_fasta_by_accession(CFG.phase3_orphan_hits_fasta)

    overlap = set(phase2_seqs) & set(phase3_seqs)
    if overlap:
        log.warning("%d accession(s) appear in both Phase 2 and Phase 3: %s", len(overlap), sorted(overlap))

    pooled = {**phase3_seqs, **phase2_seqs}
    with open(CFG.combined_candidates_fasta, "w", encoding="utf-8") as out:
        for acc, seq in pooled.items():
            out.write(f">{acc}\n{seq}\n")

log.info(
    "Pooled candidate set: %d (Phase 2 candidates + Phase 3 orphan hits, deduplicated)",
    len(pooled),
)

2026-07-22 11:28:36,879 [INFO] Pooling Phase 2 candidates and Phase 3 orphan hits ...
2026-07-22 11:28:36,934 [INFO] Pooled candidate set: 85 (Phase 2 candidates + Phase 3 orphan hits, deduplicated)


## Step 3 — Domain scan the pooled set

Runs `hmmsearch --domtblout` for all three profiles against the *combined*
pool, so the paralog and ATI screening applies to every candidate rather than
only the Phase 2 set. Each scan is independently checkpointed on its
domain-table file, and each runs through `run_command` so its full output is
captured to a log.

In [6]:
def run_hmmsearch(hmm_path: Path, target_fasta: Path, domtbl_out: Path) -> None:
    if domtbl_out.exists():
        log.info("%s exists — skipping %s scan.", domtbl_out, hmm_path.stem)
        return
    log.info("Scanning pooled candidates against %s ...", hmm_path.stem)
    run_command(
        CFG.wsl_prefix + [
            "hmmsearch", "--domtblout", domtbl_out.as_posix(),
            "-o", (CFG.out_dir / f"{hmm_path.stem}_hmmsearch.log").as_posix(),
            hmm_path.as_posix(), target_fasta.as_posix(),
        ],
        log_file=CFG.out_dir / f"{hmm_path.stem}_run.log",
    )


run_hmmsearch(CFG.pf02346_hmm, CFG.combined_candidates_fasta, CFG.domtbl_pf02346)
run_hmmsearch(CFG.pf06086_hmm, CFG.combined_candidates_fasta, CFG.domtbl_pf06086)
run_hmmsearch(CFG.pf04508_hmm, CFG.combined_candidates_fasta, CFG.domtbl_pf04508)
log.info("Domain scans complete.")

2026-07-22 11:28:36,969 [INFO] Scanning pooled candidates against PF02346 ...
2026-07-22 11:28:36,971 [INFO] Running: wsl hmmsearch --domtblout output_phase4/cand_PF02346.domtbl -o output_phase4/PF02346_hmmsearch.log PF02346.hmm output_phase4/combined_candidates.fasta
2026-07-22 11:28:37,616 [INFO] Scanning pooled candidates against PF06086 ...
2026-07-22 11:28:37,620 [INFO] Running: wsl hmmsearch --domtblout output_phase4/cand_PF06086.domtbl -o output_phase4/PF06086_hmmsearch.log PF06086.hmm output_phase4/combined_candidates.fasta
2026-07-22 11:28:37,923 [INFO] Scanning pooled candidates against PF04508 ...
2026-07-22 11:28:37,926 [INFO] Running: wsl hmmsearch --domtblout output_phase4/cand_PF04508.domtbl -o output_phase4/PF04508_hmmsearch.log PF04508.hmm output_phase4/combined_candidates.fasta
2026-07-22 11:28:38,219 [INFO] Domain scans complete.


## Step 4 — Classify & summarize

Applies the filters in order, using the shared `parse_domtblout` to read
PF06086 and PF04508 membership. The A29L motif is recorded as a `motif_present`
annotation on every entry but never changes the class. The full decision for
each accession is written to `candidates_classified.json`, with a per-class
summary logged.

In [7]:
has_pf06086 = parse_domtblout(CFG.domtbl_pf06086)
has_pf04508 = parse_domtblout(CFG.domtbl_pf04508)

classified: dict[str, dict[str, str]] = {}
for acc, seq in pooled.items():
    length = len(seq)
    motif_present = "Y" if MOTIF.search(seq) else "N"

    if acc in has_pf06086:
        entry = {"class": "A26_paralog", "reason": "PF06086 domain present"}
    elif acc in has_pf04508:
        entry = {"class": "excluded_ATIprotein", "reason": "PF04508 domain present"}
    elif length > CFG.length_threshold:
        entry = {"class": "A26_paralog", "reason": f"length {length}aa > {CFG.length_threshold}"}
    else:
        entry = {"class": "A29L_ortholog", "reason": f"PF02346-only {length}aa"}

    entry["motif_present"] = motif_present
    classified[acc] = entry

n_ortholog = sum(1 for c in classified.values() if c["class"] == "A29L_ortholog")
n_a26 = sum(1 for c in classified.values() if c["class"] == "A26_paralog")
n_ati = sum(1 for c in classified.values() if c["class"] == "excluded_ATIprotein")
n_motif_in_orth = sum(
    1 for c in classified.values()
    if c["class"] == "A29L_ortholog" and c["motif_present"] == "Y"
)

log.info("Classification results (pooled Phase 2 + Phase 3 candidates):")
log.info("  True A29L orthologs: %d (of which %d carry the A29L motif)", n_ortholog, n_motif_in_orth)
log.info("  A26 paralogs caught: %d", n_a26)
log.info("  ATI proteins caught (PF04508): %d", n_ati)

with open(CFG.classified_json, "w", encoding="utf-8") as fh:
    json.dump(classified, fh, indent=2)
log.info("DONE — %s written.", CFG.classified_json)

2026-07-22 11:28:38,354 [INFO] Classification results (pooled Phase 2 + Phase 3 candidates):
2026-07-22 11:28:38,357 [INFO]   True A29L orthologs: 83 (of which 45 carry the A29L motif)
2026-07-22 11:28:38,358 [INFO]   A26 paralogs caught: 2
2026-07-22 11:28:38,359 [INFO]   ATI proteins caught (PF04508): 0
2026-07-22 11:28:38,364 [INFO] DONE — output_phase4\candidates_classified.json written.


## Step 5 — Write true ortholog sequences

Extracts the sequences classified as `A29L_ortholog` and writes them to
`true_orthologs.fasta` — the sequence-level deliverable Phase 5 clusters. The
JSON records the decision for every candidate; this FASTA carries the actual
confirmed-ortholog sequences.

In [8]:
n_written = 0
with open(CFG.true_orthologs_fasta, "w", encoding="utf-8") as out:
    for acc in sorted(classified):
        if classified[acc]["class"] == "A29L_ortholog":
            out.write(f">{acc}\n{pooled[acc]}\n")
            n_written += 1

log.info("DONE — %d true ortholog sequence(s) written to %s", n_written, CFG.true_orthologs_fasta)

2026-07-22 11:28:38,464 [INFO] DONE — 83 true ortholog sequence(s) written to output_phase4\true_orthologs.fasta
